In [10]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser


In [9]:
from pydantic import BaseModel, Field


In [6]:
load_dotenv()
import os

In [7]:
llm1 = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    api_key=os.environ['GROQ_API_KEY1']
    # other params...
)

llm2 = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    api_key=os.environ['GROQ_API_KEY2']
    # other params...
)

llm3 = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    api_key=os.environ['GROQ_API_KEY3']
    # other params...
)

In [8]:
class JokeState(TypedDict):
    topic : str
    joke : str
    explaination : str

In [21]:
class JokeGen(BaseModel):
    joke : str = Field(description="A joke over a given topic")

In [22]:
class JokeExp(BaseModel):
    exp : str = Field(description='Explaination of joke in simple english')

In [23]:
parser1 = PydanticOutputParser(pydantic_object=JokeGen)
parser2 = PydanticOutputParser(pydantic_object=JokeExp)

In [24]:
prompt1 = PromptTemplate(
    template="""Generate a joke over the topic {topic}\n\n{format_instruction}""",
    input_variables=['topic'],
    partial_variables={
        "format_instruction": parser1.get_format_instructions()}
)

In [25]:
prompt2 = PromptTemplate(
    template="""Genearate the explaination of provide joke {joke}\n\n{format_instruction}""",
    input_variables=['joke'],
    partial_variables={
        "format_instruction": parser2.get_format_instructions()}
)

In [26]:
joke_gen = prompt1 | llm1 | parser1
joke_exp = prompt2 | llm2 | parser2

In [27]:
def generate_joke(state: JokeState):

    response = joke_gen.invoke({'topic' : state['topic']}).joke

    return {'joke': response}

In [28]:
def generate_explanation(state: JokeState):

    response = joke_exp.invoke({'joke' : state['joke']}).exp

    return {'explaination': response}

In [45]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [46]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': "Why don't pizzas ever get cold? Because they always wear crusts!",
 'explaination': "The joke plays on the word 'crust' as something a pizza 'wears' like clothing to stay warm. The crust is the outer layer of the pizza, and the humor comes from personifying the pizza as wearing it to prevent getting cold, even though a crust is part of the pizza itself."}

In [47]:
workflow.get_state(config=config1)

StateSnapshot(values={'topic': 'pizza', 'joke': "Why don't pizzas ever get cold? Because they always wear crusts!", 'explaination': "The joke plays on the word 'crust' as something a pizza 'wears' like clothing to stay warm. The crust is the outer layer of the pizza, and the humor comes from personifying the pizza as wearing it to prevent getting cold, even though a crust is part of the pizza itself."}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17b0aa-2b9e-6086-8002-777c947ee5e7'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-08T20:21:43.517607+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17b0aa-1527-6373-8001-4e0bf80e9b15'}}, tasks=(), interrupts=())

In [48]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why don't pizzas ever get cold? Because they always wear crusts!", 'explaination': "The joke plays on the word 'crust' as something a pizza 'wears' like clothing to stay warm. The crust is the outer layer of the pizza, and the humor comes from personifying the pizza as wearing it to prevent getting cold, even though a crust is part of the pizza itself."}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17b0aa-2b9e-6086-8002-777c947ee5e7'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-08T20:21:43.517607+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17b0aa-1527-6373-8001-4e0bf80e9b15'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': "Why don't pizzas ever get cold? Because they always wear crusts!"}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoi

In [49]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f17b08f-8b5d-67f6-bfff-07cd3e408766"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f17b08f-8b5d-67f6-bfff-07cd3e408766'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [50]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f17b08f-8b5f-6f0d-8000-317cbbdce429"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f17b08f-8b5f-6f0d-8000-317cbbdce429'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [51]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': "Why don't ghosts eat pasta? Because they're afraid of the ghostly garlic!",
 'explaination': "The joke plays on the word 'ghostly' which describes both the ghosts and the garlic in pasta. It's funny because ghosts can't eat pasta, but they're scared of garlic, a normal ingredient, even though they aren't physical."}

In [52]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': "Why don't pizzas ever get cold? Because they always wear crusts!", 'explaination': "The joke plays on the word 'crust' as something a pizza 'wears' like clothing to stay warm. The crust is the outer layer of the pizza, and the humor comes from personifying the pizza as wearing it to prevent getting cold, even though a crust is part of the pizza itself."}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17b0aa-2b9e-6086-8002-777c947ee5e7'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-08T20:21:43.517607+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17b0aa-1527-6373-8001-4e0bf80e9b15'}}, tasks=(), interrupts=())

In [53]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why don't pizzas ever get cold? Because they always wear crusts!", 'explaination': "The joke plays on the word 'crust' as something a pizza 'wears' like clothing to stay warm. The crust is the outer layer of the pizza, and the humor comes from personifying the pizza as wearing it to prevent getting cold, even though a crust is part of the pizza itself."}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17b0aa-2b9e-6086-8002-777c947ee5e7'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-08T20:21:43.517607+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17b0aa-1527-6373-8001-4e0bf80e9b15'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': "Why don't pizzas ever get cold? Because they always wear crusts!"}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoi

## Time Travel

In [54]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f17b08f-98a4-607e-8001-686972c3d201"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f17b08f-98a4-607e-8001-686972c3d201'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [ ]:
##So in short: that code resumes your workflow from a specific checkpoint (checkpoint_id) inside a given thread (thread_id), with no new input state (None).

workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f17b0aa-061d-610b-8000-2fee7af42b2a"}})

{'topic': 'pizza',
 'joke': 'Why did the pizza get a job? Because it wanted to make some dough!',
 'explaination': "The joke uses a pun on 'dough,' which refers to both the money earned from a job and the main ingredient in pizza. The humor comes from the unexpected idea of a pizza having a job to 'make dough,' playing on the double meaning of the word."}

In [58]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job? Because it wanted to make some dough!', 'explaination': "The joke uses a pun on 'dough,' which refers to both the money earned from a job and the main ingredient in pizza. The humor comes from the unexpected idea of a pizza having a job to 'make dough,' playing on the double meaning of the word."}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17b0ae-aa69-65c2-8003-4bb50e8d9866'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-07-08T20:23:44.187129+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17b0ae-9b67-6c83-8002-7689cdb76d16'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job? Because it wanted to make some dough!'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id'

In [59]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f17b0ae-9b67-6c83-8002-7689cdb76d16"}})

{'topic': 'pizza',
 'joke': 'Why did the pizza get a job? Because it wanted to make some dough!',
 'explaination': "The joke uses a pun on 'dough,' which means both money earned from a job and the main ingredient in pizza. The pizza is personified as wanting to work to earn money, but since it's a pizza, the dough is part of its structure. The humor comes from the unexpected twist of a pizza needing a job to make dough."}

In [60]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job? Because it wanted to make some dough!', 'explaination': "The joke uses a pun on 'dough,' which means both money earned from a job and the main ingredient in pizza. The pizza is personified as wanting to work to earn money, but since it's a pizza, the dough is part of its structure. The humor comes from the unexpected twist of a pizza needing a job to make dough."}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17b0b0-aee1-634d-8004-905297e3664d'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-07-08T20:24:38.342740+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17b0b0-9f17-6074-8003-e532ccb554e8'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job? Because it wanted to make some dough!'}, next=('generate_explanation',), config={'co

## Updating State